# siRNA Patent Landscape Pipeline
## EPO OPS API: Extraction, Comparison, Metadata Enrichment, Filtering, and Structured Data Assembly

This notebook documents the computational pipeline developed to systematically extract, filter, and structure patent data relevant to small interfering RNA (siRNA) therapeutics from the European Patent Office (EPO) Open Patent Services (OPS) database. The pipeline processes raw data from initial patent identifier retrieval through to a consolidated, analysis-ready primary activity table, supporting the broader patent landscape analysis conducted in this project.

Each section explains the rationale behind specific methodological choices, documents the relevant tools and scripts, and provides executable code to ensure reproducibility. The pipeline is modular: individual sections can be run independently, provided the input files they expect are present in the working directory.

> **API credentials:** all keys (EPO OPS and Groq) are entered **once** in the **Credentials** cell below and reused throughout the notebook. No key is hardcoded in any later cell.

### Pipeline at a glance

The eight stages run top to bottom; each reads the file(s) the previous stage wrote.

| # | Stage | Script · entry point | Reads | Writes |
|---|---|---|---|---|
| 1 | Patent-ID extraction | `epo_api_codes` / `epo_api_terms` · `download_patent_ids` | EPO OPS (live) | `..._codes_only.csv`, `..._terms_only.csv`, `..._Alnylam.csv` |
| 2 | Strategy validation | `compare_patents` · `compare` | the three ID CSVs | `..._only_in_*` / `..._shared` CSVs |
| 3 | Metadata enrichment | `epo_api_Metadata` · `fetch_biblio_from_csv` | an ID CSV | `..._metadata.csv` |
| 4 | Tier filtering | `epo_filter` · `apply_filters` | `..._metadata.csv` | `..._metadata_filtered.csv` |
| 5 | Full-text XML download | `xml_download` · `download_eps_xmls_with_ops` | an ID CSV | `eps_xmls/*.xml` · `not_in_eps.csv` |
| 6 | Table isolation | `table` · `extract_tables_from_patent` | `eps_xmls/*.xml` | `isolated_tables/*.xml` |
| 7 | XML → CSV + headers (LLM) | `xml_to_csv` · `convert_directory` | `isolated_tables/` | `csv_output/*_tables.csv` · `*_context.txt` |
| 8 | Primary-table assembly (LLM) | `xml_to_primary_table` · `build_primary_table` | `csv_output/` | `primary_table_*.csv` · IC₅₀ · viability |

Stages 1–5 use the **EPO OPS** API (credentials below); stages 7–8 use the **Groq** LLM API. Stages 5–8 are slow on free tiers (EPO enforces an 8-second delay per request; Groq rate-limits the LLM calls), so they are the long-running parts of the run.

## 0. Environment Setup

The packages below are the only dependencies; no others are required.

- **`requests`** — HTTP communication with the EPO OPS and European Publication Server (EPS) APIs.
- **`pandas`** — tabular data handling throughout the pipeline.
- **`beautifulsoup4` + `lxml`** — XML parsing during table isolation.
- **`groq`** — client for the Groq inference API (header normalisation and primary-table assembly).
- **`duckdb`** — embedded SQL engine that executes the LLM-generated `SELECT` queries during assembly.
- **`openpyxl`** — optional pandas back-end for Excel I/O, for downstream inspection.

In [2]:
%pip install requests pandas beautifulsoup4 lxml groq duckdb openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Credentials  ·  *run this first*

Every API key the pipeline needs is set **here, once**, and reused everywhere downstream.

The cell prefers **environment variables**; if one is not set it prompts you (via `getpass`), so the value is typed at run time and **never written into the notebook file** — the notebook stays safe to share or commit.

To avoid the prompts, set these before launching Jupyter:

```bash
export EPO_CONSUMER_KEY=...      export EPO_CONSUMER_SECRET=...
export GROQ_API_KEYS=key1,key2,key3,key4
```

In [18]:
import os
from getpass import getpass

def _credential(env_name, prompt):
    """Return a credential from the environment, or prompt for it (never stored)."""
    return os.getenv(env_name) or getpass(prompt)

# --- EPO OPS (Sections 1-5) ---
CONSUMER_KEY    = _credential("EPO_CONSUMER_KEY",    "EPO OPS consumer key: ")
CONSUMER_SECRET = _credential("EPO_CONSUMER_SECRET", "EPO OPS consumer secret: ")

# --- Groq (Sections 7-8). One or more keys, comma-separated. ---
# Groq free-tier limits are per ACCOUNT, so extra keys only raise throughput
# if they come from different Groq accounts.
GROQ_API_KEYS = _credential("GROQ_API_KEYS", "Groq API key(s), comma-separated: ")

print("Credentials loaded (EPO OPS + Groq).")

Credentials loaded (EPO OPS + Groq).


## 1. Patent Identifier Extraction

*Reads:* EPO OPS (live).  *Writes:* `..._codes_only.csv`, `..._terms_only.csv`, `..._only_applicant_Alnylam.csv`.

The first stage retrieves patent **family identifiers** for 2022–2025. No metadata, abstracts, or full texts are requested yet — only `Patent_ID`, `Family_ID`, and `Country`.

Two independent search strategies run in parallel; their results are expected to overlap, and that overlap is quantified in Section 2 as a validation step. A third, applicant-name query builds a high-confidence reference corpus.

### 1a. Strategy A — CPC/IPC classification codes

This strategy queries the OPS search service using Cooperative/International Patent Classification codes — examiner-assigned, structured signals of technological relevance. The primary anchor for siRNA is **`C12N 15/113`** (RNA interference / small interfering RNA); additional codes capture backbone-modification chemistry, lipid-nanoparticle delivery, and disease-specific applications.

Each code is a separate CQL query (to stay under the OPS 2,000-results-per-response limit); queries that still exceed it are auto-partitioned into monthly and then daily windows. Family-level deduplication uses a jurisdiction priority table (EP preferred over US, WO, …) so each family contributes one row.

The call below uses the `CONSUMER_KEY` / `CONSUMER_SECRET` loaded in the Credentials cell.

In [2]:
import epo_api_codes

# Optional quota check - current weekly data use vs the 4 GB free-tier limit
print("Checking EPO OPS quota...")
epo_api_codes.check_epo_quota(CONSUMER_KEY, CONSUMER_SECRET)

# CPC/IPC-based extraction across all applicants, 2022-2025
print("\nStarting CPC/IPC extraction...")
df_codes = epo_api_codes.download_patent_ids(
    consumer_key=CONSUMER_KEY,
    consumer_secret=CONSUMER_SECRET,
    start_year=2022,
    end_year=2025,
    applicant_filter=None,   # no applicant restriction - full siRNA landscape
    only_applicant=False,    # query driven by CPC/IPC codes
)
print(df_codes.head())

Checking EPO OPS quota...
      EPO OPS API - STATUS DASHBOARD

[DATA CONSUMPTION]
Weekly Volume Used: 0.0000% (0.00 MB / 4000 MB)
Quota Remaining:    100.0000%
Current API Load:   30 requests/minute

---------------------------------------------
[OK] Server clear. Safe to proceed with extraction.
---------------------------------------------

Starting CPC/IPC extraction...

=== STARTING EPO ID EXTRACTION (2022 - 2025) ===
[INFO] 6 independent query conditions.
[INFO] Exclusions active: C12N15/115 (Aptamers), C12N15/117 (Immunomodulatory)

[INFO] Processing year 2022...
  -> Query 1/6: (cpc=/low "C12N15/11" NOT (cpc="C12N15/115" OR cpc="C12N15/117"))...

[AUTH] Generating a new EPO access token...
[INFO] Year 2022: 8473 results (>2000). Slicing by month...
[INFO] Year 2022, month 1: fetching 1125 results...
[INFO] Requested: 1-100 | Received: 1-100 | Total expected: 1125
[INFO] Requested: 101-200 | Received: 101-200 | Total expected: 1125
[INFO] Requested: 201-300 | Received: 201-300 |

### 1b. Strategy B — title/abstract keywords

The second strategy matches **free-text terms** against title and abstract fields: `"siRNA"`, `"small interfering RNA"`, `"RNA interference"`, `"RNAi"`, and chemically specific backbone-modification variants.

It complements Strategy A by catching patents that are unambiguously siRNA-relevant in language but have not (yet) received the `C12N 15/113` classification — common for recent, not-yet-examined applications — and patents filed under a broader parent code. Being agnostic to examiner classification, it provides an independent recall signal. Same deep-slicing, rate-limiting, and family-deduplication logic as Strategy A; the script is `epo_api_terms.py`.

In [2]:
import epo_api_terms

print("Checking EPO OPS quota...")
epo_api_terms.check_epo_quota(CONSUMER_KEY, CONSUMER_SECRET)

print("\nStarting keyword (title/abstract) extraction...")
df_terms = epo_api_terms.download_patent_ids(
    consumer_key=CONSUMER_KEY,
    consumer_secret=CONSUMER_SECRET,
    start_year=2022,
    end_year=2025,
    applicant_filter=None,   # no applicant restriction
)
print(df_terms.head())

Checking EPO OPS quota...
      EPO OPS API - STATUS DASHBOARD

[DATA CONSUMPTION]
Weekly Volume Used: 0.9101% (36.41 MB / 4000 MB)
Quota Remaining:    99.0899%
Current API Load:   15 requests/minute

---------------------------------------------
[OK] Server clear. Safe to proceed with extraction.
---------------------------------------------

Starting keyword (title/abstract) extraction...

=== STARTING EPO ID EXTRACTION (2022 - 2025) ===
[INFO] 27 independent query conditions.
[INFO] Exclusions active: C12N15/115 (Aptamers), C12N15/117 (Immunomodulatory)

[INFO] Processing year 2022...
  -> Query 1/27: (ta=siRNA* NOT (cpc="C12N15/115" OR cpc="C12N15/117"))...

[AUTH] Generating a new EPO access token...
[INFO] Year 2022: 449 results. Fetching yearly...
[INFO] Requested: 1-90 | Received: 1-90 | Total expected: 449
[INFO] Requested: 91-180 | Received: 91-180 | Total expected: 449
[INFO] Requested: 181-270 | Received: 181-270 | Total expected: 449
[INFO] Requested: 271-360 | Received: 2

### 1c. Reference corpus — Alnylam Pharmaceuticals full portfolio

As an external validation set, the **complete Alnylam portfolio** for 2022–2025 is retrieved by querying the applicant-name field directly (`applicant_filter="Alnylam*"`, `only_applicant=True`), bypassing all CPC/IPC and keyword constraints.

Alnylam is the foremost siRNA-focused company and originator of the first approved siRNA therapeutics (like Patisiran, Givosiran, Lumasiran, Inclisiran, Vutrisiran), so essentially every family in their portfolio is siRNA-relevant. That makes it a high-confidence benchmark for the recall of Strategies A and B: any Alnylam family missing from a strategy's output is a missed relevant family for that strategy.

In [4]:
# Alnylam full portfolio - applicant-name query, no CPC/IPC filter
print("Extracting Alnylam full portfolio...")
df_alnylam = epo_api_codes.download_patent_ids(
    consumer_key=CONSUMER_KEY,
    consumer_secret=CONSUMER_SECRET,
    start_year=2022,
    end_year=2025,
    applicant_filter="Alnylam*",   # wildcard matches all Alnylam entity names
    only_applicant=True,            # name-driven query only - ignores CPC codes
)
print(df_alnylam.head())

Extracting Alnylam full portfolio...

=== STARTING EPO ID EXTRACTION (2022 - 2025) ===
[INFO] 1 independent query conditions.
[INFO] Applicant filter: Alnylam* (Only Applicant: True)

[INFO] Processing year 2022...
  -> Query 1/1: ...

[AUTH] Generating a new EPO access token...
[INFO] Year 2022: 153 results. Fetching yearly...
[INFO] Requested: 1-100 | Received: 1-100 | Total expected: 153
[INFO] Requested: 101-153 | Received: 101-153 | Total expected: 153
[INFO] Raw records accumulated (including cross-query duplicates): 153
[INFO] After family deduplication: 153 unique IDs
[INFO] Records filtered (cross-query duplicates + prior-year skips): 0
[INFO] Families tracked globally so far: 153
  [AUTOSAVE] Year 2022: 153 IDs saved to EPO_IDs_AutoSave_2022.csv

[INFO] Processing year 2023...
  -> Query 1/1: ...
[INFO] Year 2023: 181 results. Fetching yearly...
[INFO] Requested: 1-100 | Received: 1-100 | Total expected: 181
[INFO] Requested: 101-181 | Received: 101-181 | Total expected: 181


## 2. Strategy Validation — pairwise comparison of the ID sets

*Reads:* the three ID CSVs from Section 1.  *Writes:* per-comparison `only_in_*` / `shared` CSVs.

> **Dependency:** this section uses `compare_patents.py` (function `compare`), which must be present in the working directory.

With three ID sets in hand, pairwise comparison characterises their overlap and unique coverage. `compare` loads both CSVs with `dtype=str` (so numeric `Family_ID`s keep leading zeros), normalises `Family_ID` by stripping whitespace and leading zeros (the OPS API returns the same family with and without a leading zero depending on query type, which would otherwise produce false non-matches), then runs set operations to yield: families only in A, only in B, and shared. The original, non-normalised identifiers are preserved in all outputs.

| Comparison | Question |
|---|---|
| **Codes vs Keywords** | How much do the two strategies overlap, and what is each one's unique contribution? |
| **Codes vs Alnylam** | What fraction of the Alnylam corpus does the CPC/IPC strategy capture, and what does it miss? |
| **Keywords vs Alnylam** | What fraction of the Alnylam corpus does the keyword strategy capture, and what does it miss? |

### 2a · CPC/IPC codes vs keywords

In [3]:
from compare_patents import compare

# Codes (A) vs Keywords (B)
only_codes, only_terms, shared_codes_terms = compare(
    "EPO_siRNA_IDs_2022_2025_codes_only.csv",   # File A: CPC/IPC strategy output
    "EPO_siRNA_IDs_2022_2025_terms_only.csv",   # File B: keyword strategy output
)

Loading files...

  PATENT FAMILY COMPARISON  (by Family_ID)

  File A — 2025_codes_only     : 67,569 rows  |  67,569 unique families
  File B — 2025_terms_only     :  7,405 rows  |  7,405 unique families

  Only in A  (2025_codes_only) : 62,204 families  →  62,204 rows
  Only in B  (2025_terms_only) : 2,040 families  →  2,040 rows
  In both files              : 5,365 families  →  5,365 rows

  Total unique families      : 69,609


[SAVED] 62,204 rows  →  .\patents_only_in_2025_codes_only_vs_2025_terms_only.csv
[SAVED] 2,040 rows  →  .\patents_only_in_2025_terms_only_vs_2025_codes_only.csv
[SAVED] 5,365 rows  →  .\patents_common_2025_codes_only_and_2025_terms_only.csv



### 2b · CPC/IPC codes vs Alnylam portfolio

In [4]:
# Codes (A) vs Alnylam portfolio (B)
only_codes_vs_alnylam, only_alnylam_vs_codes, shared_codes_alnylam = compare(
    "EPO_siRNA_IDs_2022_2025_codes_only.csv",
    "EPO_siRNA_IDs_2022_2025_only_applicant_Alnylam.csv",
)

Loading files...

  PATENT FAMILY COMPARISON  (by Family_ID)

  File A — 2025_codes_only     : 67,569 rows  |  67,569 unique families
  File B — 2025_only_applicant_Alnylam:    316 rows  |    316 unique families

  Only in A  (2025_codes_only) : 67,264 families  →  67,264 rows
  Only in B  (2025_only_applicant_Alnylam) :    11 families  →     11 rows
  In both files              :   305 families  →    305 rows

  Total unique families      : 67,580


[SAVED] 67,264 rows  →  .\patents_only_in_2025_codes_only_vs_2025_only_applicant_Alnylam.csv
[SAVED]    11 rows  →  .\patents_only_in_2025_only_applicant_Alnylam_vs_2025_codes_only.csv
[SAVED]   305 rows  →  .\patents_common_2025_codes_only_and_2025_only_applicant_Alnylam.csv



### 2c · Keywords vs Alnylam portfolio

In [5]:
# Keywords (A) vs Alnylam portfolio (B)
only_terms_vs_alnylam, only_alnylam_vs_terms, shared_terms_alnylam = compare(
    "EPO_siRNA_IDs_2022_2025_terms_only.csv",
    "EPO_siRNA_IDs_2022_2025_only_applicant_Alnylam.csv",
)

Loading files...

  PATENT FAMILY COMPARISON  (by Family_ID)

  File A — 2025_terms_only     :  7,405 rows  |  7,405 unique families
  File B — 2025_only_applicant_Alnylam:    316 rows  |    316 unique families

  Only in A  (2025_terms_only) : 7,160 families  →  7,160 rows
  Only in B  (2025_only_applicant_Alnylam) :    71 families  →     71 rows
  In both files              :   245 families  →    245 rows

  Total unique families      : 7,476


[SAVED] 7,160 rows  →  .\patents_only_in_2025_terms_only_vs_2025_only_applicant_Alnylam.csv
[SAVED]    71 rows  →  .\patents_only_in_2025_only_applicant_Alnylam_vs_2025_terms_only.csv
[SAVED]   245 rows  →  .\patents_common_2025_terms_only_and_2025_only_applicant_Alnylam.csv



## 3. Bibliographic Metadata Enrichment

*Reads:* an ID CSV.  *Writes:* `..._metadata.csv`.

Section 1 produced lean ID lists; this stage enriches each family with full bibliographic metadata via the OPS `/biblio` endpoint. Requests are batched at 100 IDs (the `/biblio` hard limit); if a batch fails, each ID is retried individually so one bad record can't lose the batch; and if a record has no abstract, a secondary `/abstract` call is made automatically.

Fields retrieved: earliest priority date, publication date, applicant names, invention title (English preferred, with fallback), abstract (English preferred), IPC codes, and CPC codes. An adaptive rate-limiter lengthens inter-batch pauses when the server signals congestion and triggers a session cooldown after repeated throttling.

**Input choice: the Alnylam portfolio.** This run enriches the Alnylam CSV rather than the full code/keyword extracts, for two reasons: (1) the Alnylam corpus is pre-validated and high-confidence, so downstream filtering is trivial; and (2) the full landscape extracts are far larger and contain many families that prove irrelevant on inspection, enriching all of them before optimizing the filters would burn the weekly 4 GB quota for little benefit.

In [6]:
from epo_api_Metadata import fetch_biblio_from_csv

df_metadata = fetch_biblio_from_csv(
    ids_csv         = "EPO_siRNA_IDs_2022_2025_terms_only.csv",
    consumer_key    = CONSUMER_KEY,
    consumer_secret = CONSUMER_SECRET,
)
df_metadata.head()


=== STARTING EPO METADATA FETCH ===
[INFO] Input file : EPO_siRNA_IDs_2022_2025_terms_only.csv
[INFO] Patent IDs : 7405
[INFO] Batches : 75 x 100 IDs per batch

[AUTH] Generating a new EPO access token...
[INFO] Batch 1 complete — 100 records fetched so far.
[INFO] Batch 2 complete — 200 records fetched so far.
  [WARNING] Abstract fallback failed for EP.4381069.A1: ReadTimeout: HTTPSConnectionPool(host='ops.epo.org', port=443): Read timed out. (read timeout=15)
[INFO] Batch 3 complete — 300 records fetched so far.

[AUTH] Generating a new EPO access token...
[INFO] Batch 4 complete — 400 records fetched so far.
[INFO] Batch 5 complete — 500 records fetched so far.
[INFO] Batch 6 complete — 600 records fetched so far.
[INFO] Batch 7 complete — 700 records fetched so far.
[INFO] Batch 8 complete — 800 records fetched so far.
[INFO] Batch 9 complete — 900 records fetched so far.
[INFO] Batch 10 complete — 1000 records fetched so far.
[INFO] Batch 11 complete — 1100 records fetched so fa

,Patent_ID,Country,Number,Kind,Family_ID,Priority_Date,Publication_Date,Applicant,Title,Abstract,IPCs,CPCs
1550,US2022275366A1,US,2022275366,A1,83006944,20010518,20220901,SIRNA THERAPEUTICS INC [US] | SIRNA THERAPEUTI...,RNA INTERFERENCE MEDIATED INHIBITION OF GENE E...,The present invention concerns methods and rea...,,"A61K38/00, A61K47/54, A61K47/544, A61K47/549, ..."
420,US2022056441A1,US,2022056441,A1,53183180,20020220,20220224,SIRNA THERAPEUTICS INC [US] | SIRNA THERAPEUTI...,RNA INTERFERENCE MEDIATED INHIBITION OF GENE E...,The present invention concerns methods and rea...,,"C07H21/02, C12N15/111, C12N15/113, C12N15/1131..."
431,US2022112494A1,US,2022112494,A1,32046073,20020925,20220414,UNIV MASSACHUSETTS [US] | UNIVERSITY OF MASSAC...,IN VIVO GENE SILENCING BY CHEMICALLY MODIFIED ...,The present invention provides compositions fo...,,"A01K2217/075, A61K38/00, A61K48/00, C07D213/69..."
437,US2022315922A1,US,2022315922,A1,50773796,20021114,20221006,THERMO FISHER SCIENTIFIC INC [US] | Thermo Fis...,Methods and Compositions for Selecting siRNA o...,Efficient sequence specific gene silencing is ...,,"A61K31/713, A61K48/00, C12N15/1048, C12N15/111..."
682,US2022062286A1,US,2022062286,A1,27772701,20030725,20220303,UNIV SHEFFIELD [GB] | The University of Sheffield,USE OF RNAI INHIBITING PARP ACTIVITY FOR THE M...,The present invention relates to the use of an...,,"A61K31/472, A61K31/517, A61K31/5517, A61K31/70..."


## 4. Patent Classification and Tier-Based Filtering

*Reads:* `..._metadata.csv`.  *Writes:* `..._metadata_filtered.csv`.

The enriched CSV is passed through the rule-based classifier in `epo_filter.py`, which assigns every patent to one of seven mutually exclusive tiers from the co-occurrence of text signals (title/abstract) and a structural signal (the `C12N 15/113` CPC/IPC anchor). Tiers are evaluated in priority order; a patent is assigned to the highest tier it qualifies for.

No records are deleted: every patent is written out with its tier and a direct Espacenet deep-link, preserving the full dataset for transparent manual curation. The tiers guide review rather than replace it.

| Tier | Criterion | Recommended action |
|---|---|---|
| **Tier 1** | siRNA signal in **both** text **and** the `C12N15/113` anchor | Core dataset: include without further review |
| **Tier 2** | **Text only**; anchor absent (unassigned or under a broader code) | High confidence: include; CPC absence is not disqualifying |
| **Tier 3** | **Anchor only**; no text signal (abstract missing/non-English/uninformative) | Manual Espacenet review before inclusion |
| **Tier 4A** | siRNA signal **plus** a competing-tech term (aptamer, ASO, CRISPR) | Mixed-tech: review to determine primary technology |
| **Tier 4B** | Diagnostic/biomarker language; no therapeutic application | Likely out of scope for a therapeutics analysis |
| **Tier 5** | Agri/veterinary/pest-control application with a `C12N15/113` anchor | Context-dependent review (RNAi in plants/insects) |
| **Tier 6** | No text signal and no anchor | Likely irrelevant; lowest priority |
| **Tier 7** | Agri/veterinary application without an anchor | Likely irrelevant |

<br>

> **Planned improvement:** The deterministic keyword+CPC approach cannot handle negation or semantic ambiguity. A promising future iteration would explore using LLM-assisted abstract classification to reclassify the ambiguous tiers (3, 4A, and 4B). For now, the rule-based system provides sufficient signal to proceed to full-text processing at a defensible confidence threshold.

In [7]:
from epo_filter import apply_filters

input_csv  = "EPO_siRNA_IDs_2022_2025_terms_only_metadata.csv"
output_csv = "EPO_siRNA_IDs_2022_2025_terms_only_metadata_filtered.csv"

result_df = apply_filters(
    raw_data=input_csv,
    output_filename=output_csv,
    csv_sep=";",
    csv_encoding="utf-8-sig",
)
display(result_df.head(20))

[INFO] Reading CSV from disk: EPO_siRNA_IDs_2022_2025_terms_only_metadata.csv

=== STARTING PATENT CLASSIFICATION ===
[INFO] No patents will be deleted — all records go to the output CSV.

[SUCCESS] Classification complete.
  Total input patents : 7405

  TIER 1 — siRNA Confirmed (Text + CPC)             1588  ███████████████
  TIER 2 — siRNA Confirmed (Text only)              1170  ███████████
  TIER 3 — siRNA by CPC only (Check Abstract)        604  ██████
  TIER 4A — Mixed Tech (siRNA/CPC + Forbidden Term  1369  █████████████
  TIER 4B — Diagnostic/Biomarker only (Review)       145  █
  TIER 5 — Agri/Vet with siRNA CPC (Review)          361  ███
  TIER 6 — No siRNA Signal (Likely Irrelevant)      1872  ██████████████████
  TIER 7 — Agri/Vet without siRNA (Likely Irreleva   296  ██

 There are 1 patent(s) flagged as Needs_Espacenet_Review (unreadable title / missing abstract)

 There are 7 patent(s) flagged with Quantitative Efficacy Data

  Output saved to: EPO_siRNA_IDs_2022_2025_t

,Patent_ID,Priority_Date,Publication_Date,Applicant,Title,Abstract,Espacenet_Link,Tier,Has_Efficacy_Data,Needs_Espacenet_Review,IPCs,CPCs,Family_ID
0,--- TIER 1 — SIRNA CONFIRMED (TEXT + CPC) (158...,,,,,,,,,,,,
1,US2022112494A1,20020925,20220414,UNIV MASSACHUSETTS [US] | UNIVERSITY OF MASSAC...,IN VIVO GENE SILENCING BY CHEMICALLY MODIFIED ...,The present invention provides compositions fo...,https://worldwide.espacenet.com/publicationDet...,TIER 1 — siRNA Confirmed (Text + CPC),,,NaN,"A01K2217/075, A61K38/00, A61K48/00, C07D213/69...",32046073
2,US2022315922A1,20021114,20221006,THERMO FISHER SCIENTIFIC INC [US] | Thermo Fis...,Methods and Compositions for Selecting siRNA o...,Efficient sequence specific gene silencing is ...,https://worldwide.espacenet.com/publicationDet...,TIER 1 — siRNA Confirmed (Text + CPC),,,NaN,"A61K31/713, A61K48/00, C12N15/1048, C12N15/111...",50773796
3,US2022062286A1,20030725,20220303,UNIV SHEFFIELD [GB] | The University of Sheffield,USE OF RNAI INHIBITING PARP ACTIVITY FOR THE M...,The present invention relates to the use of an...,https://worldwide.espacenet.com/publicationDet...,TIER 1 — siRNA Confirmed (Text + CPC),,,NaN,"A61K31/472, A61K31/517, A61K31/5517, A61K31/70...",27772701
4,US2022119814A1,20040709,20220421,UNIV MASSACHUSETTS [US] | University of Massac...,Therapeutic alteration of transplantable tissu...,"The present invention, at least in part, relat...",https://worldwide.espacenet.com/publicationDet...,TIER 1 — siRNA Confirmed (Text + CPC),,,NaN,"A01N1/126, A61K47/6911, A61K48/005, C12N15/111...",36125793
5,US2022315945A1,20050916,20221006,MONSANTO TECHNOLOGY LLC [US] | Monsanto Techno...,Methods for genetic control of insect infestat...,The present invention relates to control of pe...,https://worldwide.espacenet.com/publicationDet...,TIER 1 — siRNA Confirmed (Text + CPC),,,NaN,"C07H21/04, C07K14/43536, C07K14/43563, C12N15/...",37497032
6,US2022042021A1,20051229,20220210,ARROWHEAD PHARMACEUTICALS INC [US] | Arrowhead...,RNAi-MEDIATED INHIBITION OF HIF1A FOR TREATMEN...,RNA interference is provided for inhibition of...,https://worldwide.espacenet.com/publicationDet...,TIER 1 — siRNA Confirmed (Text + CPC),,,NaN,"A61K31/7105, A61K31/713, A61K9/0048, A61P27/00...",38218798
7,US2022112505A1,20070615,20220414,ARROWHEAD PHARMACEUTICALS INC [US] | Arrowhead...,RNAi Inhibition of Alpha-ENaC Expression,The invention relates to compositions and meth...,https://worldwide.espacenet.com/publicationDet...,TIER 1 — siRNA Confirmed (Text + CPC),,,NaN,"A61K31/713, A61K45/06, A61P11/00, A61P11/06, A...",40130244
8,US2023053332A1,20080902,20230223,ALNYLAM PHARMACEUTICALS INC [US] | LUDWIG INST...,COMPOSITIONS AND METHODS FOR INHIBITING EXPRES...,The invention relates to a double-stranded rib...,https://worldwide.espacenet.com/publicationDet...,TIER 1 — siRNA Confirmed (Text + CPC),,,NaN,"A61K31/713, A61P35/00, C12N15/1136, C12N15/113...",41268470
9,US2022243199A1,20080925,20220804,ALNYLAM PHARMACEUTICALS INC [US] | Alnylam Pha...,Lipid formulated compositions and methods for ...,The invention relates to a double-stranded rib...,https://worldwide.espacenet.com/publicationDet...,TIER 1 — siRNA Confirmed (Text + CPC),,,NaN,"A61K31/713, A61P1/00, A61P1/04, A61P1/16, A61P...",41349261


## 5. Full-Text XML Download

*Reads:* an ID CSV (`Patent_ID`, `Family_ID`).  *Writes:* `eps_xmls/*.xml`, plus `successful_downloads.csv` and `not_in_eps.csv`.

This stage downloads **full-text XML** from the European Publication Server (EPS) — the complete specification (description, claims, and the experimental tables that record siRNA activity against target genes), which is the input to table isolation.

`xml_download.py` works at the level of the **whole patent family**, because family members are *not* copies of one another: divisionals, continuations, and even the **A** (laid-open) vs **B** (granted) versions of one application can carry different experimental data. Rather than substituting a single "best" relative, the module pulls the entire EP family and leaves the comparison of which members actually share data to a later step.

For each patent in the input CSV it:

1. **Fetches the full family** from EPO OPS — all members, all jurisdictions — via both the `/equivalents` service and the `famn=<Family_ID>` family search.
2. **Logs every non-EP member** (US, WO, JP, …) straight to `not_in_eps.csv`: only EP publications can have full text on EPS, so these are never fetched.
3. **Tests every EP member individually** on EPS (any kind code — A1, B1, … — and any publication number). Members whose XML has a real full-text structure (`<description>` / `<claims>` / `<table>`) are saved into `eps_xmls/`; every EP member with no full text is written to `not_in_eps.csv` with its ID and the reason.

Files already in `eps_xmls/` are skipped, so an interrupted run can simply be restarted and only fetches what is missing. A strict 8-second delay is enforced after every EPS and OPS request, so a full family sweep is deliberately slow.

**Outputs:** `successful_downloads.csv` (what was saved, with each member's relationship to the requested patent) and `not_in_eps.csv` (every member with no EPS full text for future analysis).

In [8]:
from xml_download import download_eps_xmls_with_ops

download_eps_xmls_with_ops(
    csv_filename     = "EPO_siRNA_IDs_2022_2025_only_applicant_Alnylam.csv",
    consumer_key     = CONSUMER_KEY,
    consumer_secret  = CONSUMER_SECRET,
    output_directory = "eps_xmls",
)

Starting FULL-FAMILY extraction for 316 patents...
Enforcing strictly 8+ second delays between all requests.
------------------------------------------------------------

Processing family of: EP4658280A2...
  Family: 3 members (1 EP number(s), 2 non-EP)
  [NO XML] EP4658280 -> no full text on EPS for any kind code (logged).

Processing family of: EP4561631A2...
  Family: 4 members (1 EP number(s), 3 non-EP)
  [NO XML] EP4561631 -> no full text on EPS for any kind code (logged).

Processing family of: EP4594492A1...
  Family: 3 members (1 EP number(s), 2 non-EP)
  [NO XML] EP4594492 -> no full text on EPS for any kind code (logged).

Processing family of: EP4547852A2...
  Family: 4 members (1 EP number(s), 3 non-EP)
  [NO XML] EP4547852 -> no full text on EPS for any kind code (logged).

Processing family of: EP4522742A2...
  Family: 4 members (1 EP number(s), 3 non-EP)
  [NO XML] EP4522742 -> no full text on EPS for any kind code (logged).

Processing family of: EP4547683A2...
  Famil

## 6. Table Isolation from Full-Text XMLs

*Reads:* `eps_xmls/*.xml`.  *Writes:* `isolated_tables/*.xml` (one file per table).

Each full-text XML is parsed with BeautifulSoup (`lxml`) to isolate the experimental tables. `table.py` locates the `EXAMPLES` heading (the standard EPO delimiter between the general description and the experimental section) and extracts every top-level `table`/`tables` element after it and within `<description>`. For each table it also captures the **five preceding paragraphs** (excluding any that contain nested tables or fall outside the description/before EXAMPLES) — typically the assay conditions, cell line, and setup needed to interpret the data. Each table plus its context is written as a self-contained XML file.

Output filenames follow `<patent_id>_table_<NN>[_T<num>][_in_vitro].xml`:

- **`<patent_id>`** — the source XML's base name (e.g. `EP2723758NWB1`).
- **`<NN>`** — a zero-padded positional index for stable ordering/uniqueness.
- **`T<num>`** — the patent's real table number parsed from the title (e.g. `T18b`); omitted when the title has no recognisable `Table <N>`.
- **`_in_vitro`** — appended when the title mentions any of: *antisense strand, cells, in vitro, sense strand, transfection, single dose, dose response, modified sequences, antisense sequence, sense sequence* — for quick downstream filtering.

To validate the end-to-end LLM steps (Sections 7–8) on a small, format-diverse subset, use the pilot list. Alternatively, you can use the cell below to process every XML file in an directory.

In [ ]:
from table import extract_tables_from_patent
import os, glob

INPUT_DIR = "eps_xmls"          # full-text XMLs downloaded in Section 5
XML_DIR   = "isolated_tables"   # one XML file per extracted table, written here

# EVERY XML downloaded in Section 5.
# patent_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.xml")))

# --- Small test
pilot = [
        "EP2999785NWB1.xml",
        "EP3960860NWA2.xml",
        "EP4365291NWA2.xml",
        "EP4744669NWA2.xml"
]

patent_files = [os.path.join(INPUT_DIR, f) for f in pilot]

for path in patent_files:
    print(f"Processing {os.path.basename(path)}...")
    extract_tables_from_patent(path, output_dir=XML_DIR)

print(f"\nTable isolation complete - {len(patent_files)} patent file(s) processed.")

Processing EP2999785NWB1.xml...
  Saved: isolated_tables\EP2999785NWB1_table_01.xml
  Saved: isolated_tables\EP2999785NWB1_table_02_T1_in_vitro.xml
  Saved: isolated_tables\EP2999785NWB1_table_03_T2_in_vitro.xml
  Saved: isolated_tables\EP2999785NWB1_table_04_T3_in_vitro.xml
  Saved: isolated_tables\EP2999785NWB1_table_05_T4_in_vitro.xml
  Saved: isolated_tables\EP2999785NWB1_table_06_T5.xml
  Saved: isolated_tables\EP2999785NWB1_table_07_T6.xml
  Saved: isolated_tables\EP2999785NWB1_table_08_T7.xml
  Saved: isolated_tables\EP2999785NWB1_table_09_T9.xml
  Saved: isolated_tables\EP2999785NWB1_table_10_T8.xml
Processing EP3960860NWA2.xml...
  Saved: isolated_tables\EP3960860NWA2_table_01_T2.xml
  Saved: isolated_tables\EP3960860NWA2_table_02_T3_in_vitro.xml
  Saved: isolated_tables\EP3960860NWA2_table_03_T4_in_vitro.xml
  Saved: isolated_tables\EP3960860NWA2_table_04_T5_in_vitro.xml
  Saved: isolated_tables\EP3960860NWA2_table_05_T6_in_vitro.xml
  Saved: isolated_tables\EP3960860NWA2_tab

## 7. XML-to-CSV Conversion and Header Normalisation

*Reads:* `isolated_tables/`.  *Writes:* `csv_output/<base>_tables.csv` + `<base>_context.txt`.

`xml_to_csv.py` turns the per-table XMLs into structured CSVs, solving two problems: extracting raw data from CALS-style XML, and producing clean, SQL-compatible column names from the heterogeneous, often multi-level headers in patent tables. Groq keys from the Credentials cell are rotated automatically to stay within free-tier limits.

Each XML yields two files in `csv_output/`:

- **`<base>_context.txt`** — title and context paragraphs from Section 6, plus any full-width annotation rows (method notes, footnotes, spanning captions) that are not column names.
- **`<base>_tables.csv`** — the normalised tabular data with SQL-compatible headers.

**Table extraction** handles three structural issues: pseudo-header rows placed in `tbody` (with `namest`/`nameend` spans) are promoted to headers; empty group-label cells are filled down from the last non-empty value (restoring cell-line/group labels that `morerows` tables leave blank); and trailing footnote `tgroup`s of full-width spanning rows are routed to the context file rather than treated as data.

**Header normalisation — two AI passes:**

- **Pass 1 — structural repair (`llama-3.3-70b-versatile`):** collapses multi-row headers, spanning group labels, and `morerows` artefacts into one flat list. The 70B model is used because sparse multi-level layouts (e.g. `"Day 3"` / `"Transfection (Hep3b)"` over `"Avg"` / `"SD"`) need reliable cross-row merging. Skipped for tables that already have a single complete header row; falls back to the rule-based `merge_multilevel_headers` if the LLM is unavailable.
- **Pass 2 — SQL normalisation (`llama-3.1-8b-instant`):** converts the clean strings to SQL identifiers (lowercase, underscores, `%`→`_pct`, `#`→`_num`, plus siRNA-specific mappings like `"IC50 (nM)"`→`ic50_nm`). Batched per file to minimise tokens; rule-based `basic_sql_normalize` fallback when the LLM is unavailable.

Three deterministic fixes follow: duplicate SQL names are de-conflicted by index; a first column of `AD-\d+` duplex IDs assigned a generic name is renamed `duplex_id`; and any `duplex`-containing name is normalised to `duplex_id`.

In [23]:
from xml_to_csv import convert_directory

convert_directory(
    "isolated_tables",
    output_dir="csv_output",
    api_keys=GROQ_API_KEYS,
)

  [Groq] 4 API key(s) loaded.

Processing: EP2999785NWB1_table_01.xml
  -> EP2999785NWB1_table_01_context.txt  (5 paragraph(s))
  SKIP tables file (no data tables found in EP2999785NWB1_table_01.xml)

Processing: EP2999785NWB1_table_02_T1_in_vitro.xml
  [Pass 2] Normalising 7 unique header(s) to SQL via Groq …
  -> EP2999785NWB1_table_02_T1_in_vitro_context.txt  (6 paragraph(s))
  -> EP2999785NWB1_table_02_T1_in_vitro_tables.csv  (1 table(s))

Processing: EP2999785NWB1_table_03_T2_in_vitro.xml
  [Pass 2] Normalising 6 unique header(s) to SQL via Groq …
  -> EP2999785NWB1_table_03_T2_in_vitro_context.txt  (6 paragraph(s))
  -> EP2999785NWB1_table_03_T2_in_vitro_tables.csv  (1 table(s))

Processing: EP2999785NWB1_table_04_T3_in_vitro.xml
  [Pass 2] Normalising 8 unique header(s) to SQL via Groq …
  -> EP2999785NWB1_table_04_T3_in_vitro_context.txt  (6 paragraph(s))
  -> EP2999785NWB1_table_04_T3_in_vitro_tables.csv  (1 table(s))

Processing: EP2999785NWB1_table_05_T4_in_vitro.xml
  [Pass

## 8. Primary Table Assembly

*Reads:* `csv_output/`.  *Writes:* `primary_table_*.csv`, `primary_ic50_table_*.csv`, `primary_cell_viability_table_*.csv` (+ `failed_tables_*` / `validation_failures_*` manifests).

The final stage consolidates the per-table CSVs into three unified schemas. `xml_to_primary_table.py` uses a second LLM pass (`llama-3.3-70b-versatile`) to generate a DuckDB `SELECT` that maps each CSV's varying columns into one fixed target schema, executed locally by the embedded DuckDB engine.

**Routing** — each CSV is assigned from keywords in its paired `_context.txt`: `IC50`/`IC 50` → the IC₅₀ table; viability keywords → the cell-viability table; everything else → the primary knockdown table.

| Output file | Content | Key fields |
|---|---|---|
| `primary_table.csv` | Knockdown activity (% inhibition at a dose) | `duplex_id`, `sense_sequence`, `antisense_sequence`, `cell_line`, `dose_nM`, `inhibition_percent` |
| `primary_ic50_table.csv` | IC₅₀ with replicate/timepoint resolution | `duplex_id`, `cell_line`, `timepoint_hrs`, `replicate`, `ic50_nM` |
| `primary_cell_viability_table.csv` | Cell-viability screen data | `duplex_id`, `cell_line`, `day`, `dose_nM`, `viability_percent` |

**Merge (primary table only)** — rows sharing `(patent_id, duplex_id, cell_line, dose_nM)` are merged: scalar annotations (sequences, oligo IDs, target gene) and the measurement fields take the first non-null value; `source_file` accumulates all contributing basenames. Sequence/oligo-only rows (no measurement) enrich matching activity rows and are then pruned, so the output has no annotation-only records. Both the modified and the unmodified sequence forms of a duplex are kept (`sense_sequence`/`antisense_sequence` hold the modified-preferred form; `*_sequence_unmodified` hold the plain form). IC₅₀ and viability tables are not merged — each row is an independent condition.

**Resilience** — the LLM-generated SQL per table is cached to disk (keyed by a SHA hash of the table content + prompt). If a run is interrupted (e.g. a sustained rate-limit), restarting reuses cached SQL for finished tables and only re-queries the API for the rest.

**Validation** — every output cell is checked: numeric fields must be numeric, doses/IC₅₀ above 10 mM (10⁷ nM) are flagged, and sequence fields must look like sequences. Offending cells are blanked and recorded in a `validation_failures` manifest rather than silently corrupting the output.

In [24]:
from xml_to_primary_table import build_primary_table

build_primary_table(
    "csv_output",
    api_keys=GROQ_API_KEYS,
    per_file_dir="per_file_output",
    file_prefixes=[
        "EP2999785NWB1",
        "EP3960860NWA2",
        "EP4365291NWA2",
        "EP4744669NWA2"]
)

  [Groq] 4 API key(s) loaded.
Found 79 table file(s) across 4 group(s) in 'csv_output'.
Detected 4 group(s): EP2999785NWB1, EP3960860NWA2, EP4365291NWA2, EP4744669NWA2. Writing one CSV set per group.

[1/4] EP2999785NWB1 — 9 tables

[EP2999785NWB1 table 1/9] Processing: EP2999785NWB1_table_02_T1_in_vitro_tables.csv
  Table type: primary
  Generating SQL...
  Executing SQL via DuckDB...
  → 88 row(s)
  → per-file CSV: per_file_output\EP2999785NWB1_table_02_T1_in_vitro_tables_primary.csv

[EP2999785NWB1 table 2/9] Processing: EP2999785NWB1_table_03_T2_in_vitro_tables.csv
  Table type: primary
  Generating SQL...
  Executing SQL via DuckDB...
  → 88 row(s)
  → per-file CSV: per_file_output\EP2999785NWB1_table_03_T2_in_vitro_tables_primary.csv

[EP2999785NWB1 table 3/9] Processing: EP2999785NWB1_table_04_T3_in_vitro_tables.csv
  Table type: primary
  Generating SQL...
  Executing SQL via DuckDB...
  → 180 row(s)
  → per-file CSV: per_file_output\EP2999785NWB1_table_04_T3_in_vitro_tables_pr

## Pipeline Overview and Next Steps

This notebook implements a complete, reproducible workflow for building a structured database of siRNA activity data from EPO patent literature, in eight sequential stages:

1. **Identifier extraction** — two independent strategies (CPC/IPC codes; title/abstract keywords) plus an Alnylam applicant-name reference query.

2. **Strategy validation** — pairwise set comparisons quantifying overlap and recall against the Alnylam reference.

3. **Metadata enrichment** — full bibliographic metadata via the OPS `/biblio` endpoint.

4. **Tier filtering** — a seven-tier rule-based classification enabling curation without information loss.

5. **Full-text XML download** — the whole EP family per patent from EPS, with everything not retrievable logged to `not_in_eps.csv`.

6. **Table isolation** — experimental tables (plus assay context) lifted from the EXAMPLES sections.

7. **XML → CSV** — heterogeneous headers normalised to consistent SQL names via a two-pass LLM strategy.

8. **Primary-table assembly** — per-table CSVs consolidated into knockdown, IC₅₀, and viability tables via LLM-generated DuckDB SQL, with merge, caching, and validation.